## 02 . Data Cleaning & Feature Engineering

--- 


#### 1. Import Libraries

In [1]:
## 1. Import libraries

import pandas as pd
import numpy as np
import os
from datetime import datetime

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

#### 2. Define Path and Create Processed Folder

In [3]:
## 2. Define Paths

RAW_PATH = "../data/raw_data/"
PROCESSED_PATH = "../data/processed/"

os.makedirs(PROCESSED_PATH, exist_ok=True)              ## Create a Folder as Processed

#### 3. Load Datasets

In [4]:
## Load All datasets and Define shape for each data have to getting rows and columns

customers = pd.read_csv(RAW_PATH + "customers.csv")
products = pd.read_csv(RAW_PATH + "products.csv")
orders = pd.read_csv(RAW_PATH + "orders.csv")
order_items = pd.read_csv(RAW_PATH + "order_items.csv")
payments = pd.read_csv(RAW_PATH + "payments.csv")
shipments = pd.read_csv(RAW_PATH + "shipments.csv")
returns = pd.read_csv(RAW_PATH + "returns.csv")
reviews = pd.read_csv(RAW_PATH + "customer_reviews.csv")

print(f"customers     : {customers.shape}")
print(f"products      : {products.shape}")
print(f"orders        : {orders.shape}")
print(f"order_items   : {order_items.shape}")
print(f"payments      : {payments.shape}")
print(f"shipments     : {shipments.shape}")
print(f"returns       : {returns.shape}")
print(f"reviews       : {reviews.shape}")

customers     : (25000, 17)
products      : (550, 14)
orders        : (100000, 14)
order_items   : (254331, 9)
payments      : (100000, 8)
shipments     : (89681, 11)
returns       : (12075, 8)
reviews       : (77530, 8)


#### 4. Data Cleaning

In [7]:
## Date Cleaning

print("\n" + "="*60)
print("STARTING DATA CLEANING")
print("="*60)

# ----- 4.1 Convert Date Columns -----

date_columns = {
    'customers': ['customer_signup_date', 'last_order_date'],
    'products': ['product_launch_date'],
    'orders': ['order_date'],
    'payments': ['payment_date'],
    'shipments': ['dispatch_date', 'expected_delivery_date', 'actual_delivery_date'],
    'returns': ['return_date'],
    'reviews': ['review_date']
}

for table_name, cols in date_columns.items():
    df = globals()[table_name]
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col],errors='coerce',dayfirst=True)           ## Using this Column converted in Datetime and if columns have null then it represent NaT.


print("Date columns converted successfully.")
# ----- 4.2 Handle Missing Values -----

# Customers - age
customers['age'] = customers['age'].fillna(customers['age'].median())

# Customers - preferred_device
customers['preferred_device'] = customers['preferred_device'].fillna("Unknown")

# products - rating_average
products['rating_average'] = products['rating_average'].fillna(products['rating_average'].median())

# Orders - coupon_code
orders['coupon_code'] = orders['coupon_code'].fillna('No Coupon')

## Customer reviews
reviews['review_sentiment'] = reviews['review_sentiment'].fillna(reviews['rating'].map
                                            ({
                                               1: 'Negative',
                                               2: 'Negative',
                                               3: 'Neutral',
                                               4: 'Positive',
                                               5: 'Positive' 
                                            }))  

print("Missing values handled")                                                                                 

# ----- 4.3 Remove Duplicates (if any) -----

customers = customers.drop_duplicates(subset=['customer_id'])
products = products.drop_duplicates(subset=['product_id'])
orders = orders.drop_duplicates(subset=['order_id'])
order_items = order_items.drop_duplicates(subset=['order_item_id'])
payments = payments.drop_duplicates(subset=['payment_id'])
shipments = shipments.drop_duplicates(subset=['shipment_id'])
returns = returns.drop_duplicates(subset=['return_id'])
reviews = reviews.drop_duplicates(subset=['review_id'])




STARTING DATA CLEANING
Date columns converted successfully.
Missing values handled


In [15]:
# 5. Feature Engineering

print("\n" + "="*60)
print("STARTING FEATURE ENGINEERING")
print("="*60)

orders['order_year'] = orders['order_date'].dt.year
orders['order_month'] = orders['order_date'].dt.month
orders['order_quarter'] = orders['order_date'].dt.quarter
orders['order_weekday'] = orders['order_date'].dt.day_name()
orders['is_weekend'] = orders['order_date'].dt.day_of_week >=5

print("Time featured created on orders.")

# ----- 5.2 Shipments: Delivery Delay -----

shipments['delivery_delay_days'] = (
    shipments['actual_delivery_date'] - shipments['expected_delivery_date']).dt.days
shipments['delivery_delay_days'] = shipments['delivery_delay_days'].fillna(0)           ## Fill null delay as 0 (for cancelled/ not delivered)

print("Delivery delay feature created")
# ----- 5.3 Order Items: Profit Margin -----

order_items['profit_margin'] = np.where(
    order_items['item_revenue'] > 0,
    (order_items['profit'] / order_items['item_cost'] ) * 100,
    0                                                                                       ## Profit margin calculates using profit,cost and revenue.
)

print("Profit margin feature created")

# ----- 5.4 Customer Tenure (in days) -----

max_date = orders['order_date'].max()
customers['customer_tenure_days'] = (max_date - customers['customer_signup_date']).dt.days          ## How long customers associated with the business

print("customers tenure feature are created")

# ----- 5.5 Create Flag: Has the customer ever returned a product? -----

returned_customers = returns['customer_id'].unique()
customers['has_returned'] = customers['customer_id'].isin(returned_customers).astype(int)

print("has_returned flag created")



STARTING FEATURE ENGINEERING
Time featured created on orders.
Delivery delay feature created
Profit margin feature created
customers tenure feature are created
has_returned flag created


In [17]:
# 6. Create Master Fact Table

print("\nCreating Master Order Fact Table...")

master = (
    order_items
    .merge(orders, on='order_id', how='left')
    .merge(products, on='product_id', how='left')
    .merge(customers[['customer_id', 'customer_segment', 'state', 'city',
                        'loyalty_tier', 'customer_status', 'gender', 'age_group',
                        'has_returned', 'customer_tenure_days']],
           on='customer_id', how='left')
)

print(f"Master table shape: {master.shape}")


Creating Master Order Fact Table...
Master table shape: (254331, 50)


In [20]:
# 7. Save Cleaned Data

print("\n" + "="*60)
print("SAVING CLEANED FILES")
print("="*60)

customers.to_csv(PROCESSED_PATH + "customers_clean.csv", index=False)
products.to_csv(PROCESSED_PATH + "products_clean.csv", index=False)
orders.to_csv(PROCESSED_PATH + "orders_clean.csv", index=False)
order_items.to_csv(PROCESSED_PATH + "order_items_clean.csv", index=False)
payments.to_csv(PROCESSED_PATH + "payments_clean.csv", index=False)
shipments.to_csv(PROCESSED_PATH + "shipments_clean.csv", index=False)
returns.to_csv(PROCESSED_PATH + "returns_clean.csv", index=False)
reviews.to_csv(PROCESSED_PATH + "reviews_clean.csv", index=False)
master.to_csv(PROCESSED_PATH + "master_order_fact.csv", index=False)

print("✓ All cleaned files saved successfully in data/processed/")


SAVING CLEANED FILES
✓ All cleaned files saved successfully in data/processed/
